[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C18_Computer_Vision_Course/03_detection/03_detection.ipynb)

# 03 · 目标检测（纯 numpy 从零）

目标：把检测的全套评测口径从零写出来——**IoU**、**NMS（贪心，逐类）**、**TP/FP 匹配**、**precision-recall 曲线**、**AP（单调包络下面积）**、**mAP**，外加 **anchor 的 IoU 匹配**，每步 `assert` 验证性质。

**路线**：
1. IoU：交集 `max(0,...)`、对称、尺度无关
2. NMS：贪心去重，逐类
3. 匹配 TP/FP：按置信度降序、每个 GT 最多匹配一次
4. PR 曲线（累积计数）
5. AP（单调包络 + 面积）与 mAP
6. anchor 的 IoU 匹配
7. ✏️ 练习 → 📖 答案 → 🧪 真实 optdigits 检测胶囊

> **本课纪律**：IoU 交集必须夹 `max(0,...)`；匹配按置信度降序、每 GT 一次；AP 用单调包络。

## 1 · IoU：交并比

框用 `(x1,y1,x2,y2)`。交集矩形边界 `[max(x1),min(x2)]×[max(y1),min(y2)]`，**某维 max(起)>=min(止) 则不相交、交集为 0**（这就是必须 `max(0,...)` 的原因）。
IoU = 交 / (面积A + 面积B - 交)。验证对称、尺度无关、∈[0,1]。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def box_area(b):
    return max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)   # 必须 max(0,...)
    inter = iw * ih
    union = box_area(a) + box_area(b) - inter
    return inter / union if union > 0 else 0.0

A = (0, 0, 2, 2); B = (1, 1, 3, 3)
# 交集 = [1,2]x[1,2] = 1; 并集 = 4+4-1 = 7
print('IoU(A,B) =', round(iou(A, B), 4), '(应 = 1/7 ≈ 0.1429)')
assert abs(iou(A, B) - 1/7) < 1e-9
# 对称
assert iou(A, B) == iou(B, A), 'IoU 应对称'
# 不相交 -> 0
assert iou((0,0,1,1), (5,5,6,6)) == 0.0, '不相交应为 0'
# 完全重合 -> 1; 尺度无关
assert abs(iou(A, A) - 1.0) < 1e-9
assert abs(iou((0,0,2,2),(1,1,3,3)) - iou((0,0,20,20),(10,10,30,30))) < 1e-9, '同比例缩放 IoU 不变'
print('✅ IoU 正确：对称、不相交=0、自身=1、尺度无关')

## 2 · NMS：贪心去重

按置信度降序：取最高分框→保留→删掉与它 IoU > 阈值的框→重复。
验证：一堆重叠框被压成一个，远处的独立框被保留。

In [ ]:
def nms(boxes, scores, iou_thresh=0.5):
    '''返回保留框的下标(按分数降序)。'''
    idxs = list(np.argsort(-np.asarray(scores)))      # 分数从高到低
    keep = []
    while idxs:
        i = idxs.pop(0)                                # 当前最高分
        keep.append(i)
        idxs = [j for j in idxs if iou(boxes[i], boxes[j]) <= iou_thresh]  # 删重叠
    return keep

# 3 个几乎重叠的框(同物体) + 1 个远处独立框
boxes = [(0,0,10,10), (1,1,11,11), (2,0,12,10), (50,50,60,60)]
scores = [0.9, 0.8, 0.85, 0.7]
keep = nms(boxes, scores, iou_thresh=0.5)
print('保留的框下标:', keep)
assert 0 in keep, '最高分框应保留'
assert 3 in keep, '远处独立框应保留'
assert len(keep) == 2, '3 个重叠框应压成 1 个, 加独立框共 2'
print('✅ NMS 把重叠框压成一个、保留独立框')

In [ ]:
def nms_per_class(boxes, scores, labels, iou_thresh=0.5):
    '''逐类 NMS：不同类的框即使重叠也不互相抑制。'''
    keep = []
    for cls in set(labels):
        idx = [i for i, l in enumerate(labels) if l == cls]
        sub_keep = nms([boxes[i] for i in idx], [scores[i] for i in idx], iou_thresh)
        keep += [idx[k] for k in sub_keep]
    return sorted(keep)

# 同位置两个框, 不同类 -> 都应保留
b2 = [(0,0,10,10), (0,0,10,10)]; s2 = [0.9, 0.8]; l2 = ['cat', 'dog']
k2 = nms_per_class(b2, s2, l2, 0.5)
assert len(k2) == 2, '不同类的重叠框不应互相抑制'
print('✅ 逐类 NMS：不同类的重叠框都保留')

## 3 · 匹配预测与真值：TP / FP

按置信度**降序**处理每个预测框，与**尚未被匹配**的 GT 算 IoU；最大 IoU≥阈值→TP 并占用该 GT，否则→FP。
纪律：每个 GT 最多被匹配一次（重复命中算 FP）。

In [ ]:
def match_detections(pred_boxes, pred_scores, gt_boxes, iou_thresh=0.5):
    '''返回 (tp, fp) 二值数组(按分数降序), 以及命中的 GT 数。'''
    order = np.argsort(-np.asarray(pred_scores))
    matched_gt = set()
    tp = np.zeros(len(order)); fp = np.zeros(len(order))
    for rank, pi in enumerate(order):
        best_iou, best_gt = 0.0, -1
        for gi, gb in enumerate(gt_boxes):
            if gi in matched_gt:
                continue                              # 该 GT 已被占用
            v = iou(pred_boxes[pi], gb)
            if v > best_iou:
                best_iou, best_gt = v, gi
        if best_iou >= iou_thresh and best_gt >= 0:
            tp[rank] = 1; matched_gt.add(best_gt)
        else:
            fp[rank] = 1
    return tp, fp, len(matched_gt)

gt = [(0,0,10,10), (50,50,60,60)]
# 预测: 命中gt0(高分), 重复命中gt0(应FP), 命中gt1
pb = [(0,0,10,10), (1,1,11,11), (50,50,60,60)]
ps = [0.9, 0.8, 0.7]
tp, fp, nhit = match_detections(pb, ps, gt, 0.5)
print('tp:', tp.astype(int), 'fp:', fp.astype(int), '命中GT:', nhit)
assert tp.sum() == 2 and fp.sum() == 1, '应 2 TP(各GT一次) + 1 FP(重复命中)'
assert nhit == 2, '两个 GT 各被命中一次'
print('✅ 匹配正确：每个 GT 只算一次 TP，重复命中算 FP')

## 4 · Precision-Recall 曲线（累积计数）

把 TP/FP 按分数降序**累加**，每一步算 precision=累积TP/(累积TP+累积FP)、recall=累积TP/总GT。
随阈值放宽，recall 单调上升。

In [ ]:
def pr_curve(tp, fp, n_gt):
    ctp = np.cumsum(tp); cfp = np.cumsum(fp)
    precision = ctp / (ctp + cfp + 1e-12)
    recall = ctp / (n_gt + 1e-12)
    return precision, recall

# 构造一组检测：5 个 GT
tp = np.array([1,1,0,1,1,0,1.])    # 按分数降序的 TP 序列
fp = 1 - tp
prec, rec = pr_curve(tp, fp, n_gt=5)
print('precision:', np.round(prec, 3))
print('recall:   ', np.round(rec, 3))
assert np.all(np.diff(rec) >= -1e-12), 'recall 应单调不减(累积)'
assert abs(rec[-1] - 5/5) < 1e-9, '全部 5 个 TP -> 末端 recall=1'
assert (prec <= 1.0 + 1e-9).all() and (prec >= 0).all()
print('✅ PR 曲线：recall 单调上升、precision∈[0,1]')

## 5 · AP（单调包络 + 面积）与 mAP

AP = PR 曲线下面积。先把 precision 做成**单调不增的包络**（每点取其右侧最大 precision），再算曲线下面积。
mAP = 各类 AP 的平均。验证：完美检测 AP=1。

In [ ]:
def average_precision(tp, fp, n_gt):
    '''全点插值 AP：单调包络 precision, 对 recall 积分(面积)。'''
    prec, rec = pr_curve(tp, fp, n_gt)
    # 在两端补点便于积分
    mrec = np.concatenate(([0.0], rec, [rec[-1]]))
    mpre = np.concatenate(([0.0], prec, [0.0]))
    # 单调包络: 从右往左取累积最大
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    # recall 变化处的面积之和
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))

# 完美检测: 全 TP -> AP = 1
ap_perfect = average_precision(np.ones(5), np.zeros(5), 5)
print('完美检测 AP =', round(ap_perfect, 4))
assert abs(ap_perfect - 1.0) < 1e-9, '全 TP 应 AP=1'
# 有错检的 AP 应 < 1 但 > 0
ap_mixed = average_precision(np.array([1,0,1,1,0,1.]), np.array([0,1,0,0,1,0.]), 5)
print('混合检测 AP =', round(ap_mixed, 4))
assert 0 < ap_mixed < 1, '有错检 AP 应在 (0,1)'
# 漏检(只命中部分 GT): recall 上不到 1, AP 更低
ap_miss = average_precision(np.array([1,1.]), np.array([0,0.]), 5)
assert ap_miss < ap_perfect, '漏检应降低 AP'
print('✅ AP 正确：完美=1、混合∈(0,1)、漏检降低 AP')

In [ ]:
def mean_ap(per_class_tpfp, iou_thresh=0.5):
    '''per_class_tpfp: {cls: (tp, fp, n_gt)} -> mAP。'''
    aps = [average_precision(tp, fp, n) for (tp, fp, n) in per_class_tpfp.values()]
    return float(np.mean(aps)), aps

pc = {
    'cat': (np.array([1,1,0,1.]), np.array([0,0,1,0.]), 3),
    'dog': (np.array([1,0,1.]),   np.array([0,1,0.]),   2),
}
mAP, aps = mean_ap(pc)
print('per-class AP:', [round(a,3) for a in aps], '-> mAP =', round(mAP, 3))
assert 0 < mAP <= 1 and abs(mAP - np.mean(aps)) < 1e-12
print('✅ mAP = 各类 AP 的平均')

## 6 · anchor 的 IoU 匹配

训练时给每个 anchor 分配标签：与某 GT IoU≥0.7→正样本（学该物体）；<0.3→负（背景）；中间忽略。
验证：贴合 GT 的 anchor 是正、远离的是负。

In [ ]:
def assign_anchors(anchors, gt_boxes, pos_thr=0.7, neg_thr=0.3):
    '''返回每个 anchor 的标签: 1=正, 0=负, -1=忽略; 及匹配的 GT 下标。'''
    labels = np.full(len(anchors), -1, dtype=int)
    gt_idx = np.full(len(anchors), -1, dtype=int)
    for ai, a in enumerate(anchors):
        ious = [iou(a, g) for g in gt_boxes] if gt_boxes else [0.0]
        best = int(np.argmax(ious)); v = ious[best]
        if v >= pos_thr:
            labels[ai] = 1; gt_idx[ai] = best
        elif v < neg_thr:
            labels[ai] = 0
        # 否则保持 -1(忽略)
    return labels, gt_idx

gt = [(10, 10, 30, 30)]
anchors = [(10,10,30,30), (11,11,31,31), (12,12,28,28), (100,100,120,120), (20,20,45,45)]
labels, gidx = assign_anchors(anchors, gt)
print('anchor 标签:', labels, '(1正/0负/-1忽略)')
assert labels[0] == 1, '完全贴合的 anchor 应为正'
assert labels[3] == 0, '远处 anchor 应为负'
assert (labels == 1).sum() >= 1 and (labels == 0).sum() >= 1
print('✅ anchor 匹配：贴合GT=正、远离=负、中间忽略')

---
## ✏️ 练习 1：从零实现 IoU

实现 `my_iou(a, b)`（框 `(x1,y1,x2,y2)`）：注意交集宽高要 `max(0,...)`，不相交返回 0。

In [ ]:
def my_iou(a, b):
    # TODO: 交集 [max(x1),min(x2)]x[max(y1),min(y2)]; iw,ih 用 max(0,...);
    #   inter=iw*ih; union=areaA+areaB-inter; return inter/union (union>0 else 0)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(my_iou((0,0,2,2),(1,1,3,3)) - 1/7) < 1e-9
assert my_iou((0,0,1,1),(2,2,3,3)) == 0.0, '不相交=0'
assert abs(my_iou((0,0,4,4),(0,0,4,4)) - 1.0) < 1e-9, '自身=1'
assert my_iou((0,0,2,2),(1,1,3,3)) == my_iou((1,1,3,3),(0,0,2,2)), '对称'
print('✅ 练习 1 通过：IoU 正确')

## ✏️ 练习 2：贪心 NMS

实现 `my_nms(boxes, scores, thr)`：按分数降序，保留最高分、删与它 IoU>thr 的，返回保留下标列表。

In [ ]:
def my_nms(boxes, scores, thr=0.5):
    # TODO: idxs=按 -scores 排序; while idxs: 取首个加入 keep; 过滤掉与它 IoU>thr 的
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
boxes = [(0,0,10,10),(1,1,11,11),(50,50,60,60)]; scores=[0.9,0.8,0.7]
keep = my_nms(boxes, scores, 0.5)
assert 0 in keep and 2 in keep, '最高分框与独立框应保留'
assert 1 not in keep, '与最高分重叠的框应被删'
assert len(keep) == 2
print('✅ 练习 2 通过：NMS 去重正确')

## ✏️ 练习 3：AP（单调包络面积）

实现 `my_ap(precision, recall)`：给已排好的 precision/recall 数组，先把 precision 做成右侧累积最大的包络，再算 `Σ (recall[i]-recall[i-1]) * precision_envelope[i]`。

In [ ]:
def my_ap(precision, recall):
    # TODO:
    #   mrec=[0]+recall+[recall[-1]]; mpre=[0]+precision+[0]
    #   从右往左: mpre[i]=max(mpre[i],mpre[i+1])  (单调包络)
    #   idx=recall 变化处; return sum((mrec[idx+1]-mrec[idx])*mpre[idx+1])
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 完美 PR: precision 全 1, recall 0.2..1.0 -> AP=1
prec = np.array([1.,1.,1.,1.,1.]); rec = np.array([.2,.4,.6,.8,1.])
assert abs(my_ap(prec, rec) - 1.0) < 1e-9, '完美 PR 应 AP=1'
# 阶梯下降
prec2 = np.array([1.,0.5,0.67,0.5]); rec2 = np.array([.25,.25,.5,.5])
ap2 = my_ap(prec2, rec2)
assert 0 < ap2 <= 1, 'AP 应在 (0,1]'
print(f'✅ 练习 3 通过：AP 正确 (完美=1, 阶梯={ap2:.3f})')

## ✏️ 练习 4：anchor 正负样本判定

实现 `anchor_label(anchor, gt_boxes, pos=0.7, neg=0.3)`：返回 1(最大IoU≥pos) / 0(<neg) / -1(中间)。

In [ ]:
def anchor_label(anchor, gt_boxes, pos=0.7, neg=0.3):
    # TODO: best=max(iou(anchor,g) for g in gt_boxes); 
    #   if best>=pos: 1; elif best<neg: 0; else -1
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
gt = [(10,10,30,30)]
# IoU 验算: (12,12,32,32)vs(10,10,30,30): 交=18x18=324, 并=400+400-324=476, IoU≈0.68 -> 忽略
assert anchor_label((10,10,30,30), gt) == 1, '完全重合=正'
assert anchor_label((100,100,120,120), gt) == 0, '远离=负'
assert anchor_label((12,12,32,32), gt) == -1, 'IoU≈0.68(在0.3~0.7间)=忽略'
print('✅ 练习 4 通过：anchor 正负判定正确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def my_iou(a, b):
    ix1, iy1 = max(a[0],b[0]), max(a[1],b[1])
    ix2, iy2 = min(a[2],b[2]), min(a[3],b[3])
    iw, ih = max(0.0, ix2-ix1), max(0.0, iy2-iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.0

In [ ]:
# 练习 2 参考答案
def my_nms(boxes, scores, thr=0.5):
    idxs = list(np.argsort(-np.asarray(scores)))
    keep = []
    while idxs:
        i = idxs.pop(0); keep.append(i)
        idxs = [j for j in idxs if iou(boxes[i], boxes[j]) <= thr]
    return keep

In [ ]:
# 练习 3 参考答案
def my_ap(precision, recall):
    mrec = np.concatenate(([0.0], recall, [recall[-1]]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(len(mpre)-2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i+1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx+1] - mrec[idx]) * mpre[idx+1]))

In [ ]:
# 练习 4 参考答案
def anchor_label(anchor, gt_boxes, pos=0.7, neg=0.3):
    best = max((iou(anchor, g) for g in gt_boxes), default=0.0)
    if best >= pos: return 1
    if best < neg:  return 0
    return -1

---
## 🧪 真实数据胶囊：optdigits 上的玩具检测 + mAP

用真实 optdigits 数字图，把每张数字**贴到一张大画布的随机位置**当作「物体」，用模板匹配出候选框、跑完整 NMS + mAP 评测。机制全部复用前面写的函数。

In [ ]:
def load_digits_or_synth(n=20, seed=0):
    try:
        from sklearn.datasets import load_digits
        d = load_digits(); return d.images[:n].astype(float)/16.0, d.target[:n].astype(int), 'real optdigits'
    except Exception:
        r = np.random.default_rng(seed)
        X = np.zeros((n,8,8)); y = r.integers(0,10,n)
        for i in range(n):
            cy,cx = r.integers(2,6,2); X[i, cy-1:cy+2, cx-1:cx+2] = 1.0
        return X, y, 'synthetic fallback'

digs, labs, src = load_digits_or_synth(8)
print('source:', src)
# 在 40x40 画布上放 3 个数字, 记录 GT 框
r = np.random.default_rng(1)
canvas = np.zeros((40, 40)); gt_boxes = []
for k in range(3):
    yy, xx = int(r.integers(0, 32)), int(r.integers(0, 32))
    canvas[yy:yy+8, xx:xx+8] = np.maximum(canvas[yy:yy+8, xx:xx+8], digs[k])
    gt_boxes.append((xx, yy, xx+8, yy+8))
# 模板匹配生成候选(在每个亮区附近), 这里直接用 GT 加扰动模拟检测器输出
pred_boxes, pred_scores = [], []
for (x1,y1,x2,y2) in gt_boxes:
    pred_boxes.append((x1+r.integers(-1,2), y1+r.integers(-1,2), x2+r.integers(-1,2), y2+r.integers(-1,2)))
    pred_scores.append(0.9)
    pred_boxes.append((x1+5, y1+5, x2+5, y2+5)); pred_scores.append(0.4)  # 偏移的低分误检
# NMS -> 匹配 -> AP
keep = nms(pred_boxes, pred_scores, 0.5)
kb = [pred_boxes[i] for i in keep]; ks = [pred_scores[i] for i in keep]
tp, fp, nhit = match_detections(kb, ks, gt_boxes, 0.5)
ap = average_precision(tp, fp, len(gt_boxes))
print(f'GT {len(gt_boxes)} 个, NMS 后候选 {len(kb)} 个, 命中 {nhit}, AP={ap:.3f}')
assert len(gt_boxes) == 3
assert 0.0 <= ap <= 1.0, 'AP 应在 [0,1]'
assert nhit >= 2, '高分准框应命中多数 GT'
print('✅ 真实数字检测全管线(NMS+匹配+AP)跑通')

**🧪 胶囊练习**：实现 `detect_ap(gt_boxes, pred_boxes, pred_scores, iou_thr=0.5)`：跑 NMS→匹配→AP，返回 AP。

In [ ]:
def detect_ap(gt_boxes, pred_boxes, pred_scores, iou_thr=0.5):
    # TODO: keep=nms(...); 取子集; tp,fp,_=match_detections(...); return average_precision(tp,fp,len(gt_boxes))
    raise NotImplementedError

In [ ]:
# 自测
ap = detect_ap(gt_boxes, pred_boxes, pred_scores, 0.5)
assert 0.0 <= ap <= 1.0
print(f'✅ 胶囊练习通过：detect_ap = {ap:.3f}')

In [ ]:
# 📖 胶囊参考答案
def detect_ap(gt_boxes, pred_boxes, pred_scores, iou_thr=0.5):
    keep = nms(pred_boxes, pred_scores, iou_thr)
    kb = [pred_boxes[i] for i in keep]; ks = [pred_scores[i] for i in keep]
    tp, fp, _ = match_detections(kb, ks, gt_boxes, iou_thr)
    return average_precision(tp, fp, len(gt_boxes))

### 小结
- **IoU**：交集宽高用 `max(0,...)`；对称、尺度无关、∈[0,1]。检测一切评测的原子。
- **NMS**：贪心去重，逐类进行（不同类不互相抑制）。
- **匹配 TP/FP**：按置信度降序、每个 GT 最多匹配一次（重复命中=FP）。
- **PR 曲线**：累积 TP/FP，recall 单调上升。
- **AP/mAP**：AP=单调包络下面积；mAP=各类 AP 平均。比较时口径(IoU 阈值/插值)必须一致。
- **anchor**：用 IoU 把 anchor 分成正/负/忽略，把定位变成「选参考框 + 微调」。

下一站：**模块 04 · 图像分割** —— 把粒度从「框」推到「每个像素」(Dice/mIoU/连通域)。